# BiLSTM SMILES Encoder — Hyperparameter Sweep Results

This notebook visualizes and analyzes the results generated by `experiments/bilstm_encoder_sweep.py`, saved in `outputs/encoders/BiLSTM/sweep/`.

Hyperparameter configurations are compared using validation RMSE, with learning curves showing training behavior and multiple-seed verification assessing consistency. The sweep uses validation data only.

In [ ]:

%matplotlib inline

from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate the AI project directory.
AI_DIR = Path.cwd()
if AI_DIR.name == "notebooks":
    AI_DIR = AI_DIR.parent
elif (AI_DIR / "ai" / "outputs").exists():
    AI_DIR = AI_DIR / "ai"

SWEEP_DIR = AI_DIR / "outputs" / "encoders" / "BiLSTM" / "sweep"
HISTORY_DIR = SWEEP_DIR / "epoch_history"

print("Sweep directory:", SWEEP_DIR)
print("Directory exists:", SWEEP_DIR.exists())
print("CSV files:", [p.name for p in sorted(SWEEP_DIR.glob("*.csv"))])
print("Epoch histories:", len(list(HISTORY_DIR.glob("*.csv"))))


## Hyperparameter Sweep Analysis

Stages 2–7 evaluate architecture, regularization, and training hyperparameters. Each experiment is compared using its best validation RMSE.

In [ ]:

summary_files = sorted(SWEEP_DIR.glob("*.csv"))

if summary_files:
    results = pd.concat(
        [
            pd.read_csv(path).assign(stage=path.stem)
            for path in summary_files
        ],
        ignore_index=True,
    )

    print(f"Loaded {len(results)} completed runs.")
    print("Columns:", results.columns.tolist())
    display(results.head())
else:
    results = pd.DataFrame()
    print("No sweep results found.")


In [ ]:

stage_settings = {
    "2-depth": ["NUM_LAYERS"],
    "3-embedding": ["EMBEDDING_DIM"],
    "4-hidden": ["HIDDEN_DIM"],
    "5-regularization": ["DROPOUT", "WEIGHT_DECAY"],
    "6-learning-rate": ["LR"],
    "7-batch-size": ["BATCH_SIZE"],
}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for ax, (stage, params) in zip(axes.flat, stage_settings.items()):
    data = results[results["stage"] == stage].copy()

    if data.empty:
        ax.set_title(f"{stage} (no results)")
        ax.axis("off")
        continue

    data["label"] = data["cfg"].apply(
        lambda s: ", ".join(
            f"{key}={json.loads(s)[key]}" for key in params
        )
    )

    data = data.sort_values("val_RMSE")
    ax.barh(data["label"], data["val_RMSE"])
    ax.invert_yaxis()
    ax.set_title(stage)
    ax.set_xlabel("Best Validation RMSE")
    ax.grid(axis="x", alpha=0.2)

plt.tight_layout()
plt.show()


## Finalist Comparison

The strongest configurations from the exploratory sweep were evaluated with longer training schedules of up to 30 epochs. 

In [ ]:

finalists = results[
    results["stage"] == "8-finalists"
].copy()

finalists = finalists.sort_values("val_RMSE")

if not finalists.empty:
    finalists["label"] = finalists["cfg"].apply(
        lambda s: (
            f"Hidden {json.loads(s)['HIDDEN_DIM']}, "
            f"WD {json.loads(s)['WEIGHT_DECAY']}"
        )
    )

    display(finalists[
        ["label", "val_RMSE", "secs", "best_epoch"]
    ])

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(finalists["label"], finalists["val_RMSE"])
    ax.invert_yaxis()
    ax.set_xlabel("Best Validation RMSE")
    ax.set_title("Stage 8: Finalist Comparison")
    ax.grid(axis="x", alpha=0.2)

    plt.tight_layout()
    plt.show()
else:
    print("No Stage 8 results available.")



## Seed Verification

Stage 9 evaluates the two leading configurations across multiple random seeds, with a maximum of 40 epochs and early stopping patience of 10. Mean validation RMSE is used to compare overall performance, while standard deviation indicates consistency across runs.


In [ ]:

seeds = results[
    results["stage"] == "9-seed-verification"
].copy()

if not seeds.empty:
    summary = (
        seeds.groupby("cfg")["val_RMSE"]
        .agg(["mean", "std", "count"])
        .sort_values("mean")
        .reset_index()
    )

    summary["label"] = summary["cfg"].apply(
        lambda s: (
            f"WD {json.loads(s)['WEIGHT_DECAY']}"
        )
    )

    display(summary[
        ["label", "mean", "std", "count"]
    ])

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(
        summary["label"],
        summary["mean"],
        yerr=summary["std"].fillna(0),
        capsize=5
    )
    ax.set_ylabel("Mean Validation RMSE")
    ax.set_title("Stage 9: Performance Across Random Seeds")
    ax.grid(axis="y", alpha=0.2)

    plt.tight_layout()
    plt.show()
else:
    print("No Stage 9 results available.")



## Learning Curves

Training and validation performance are compared across epochs to examine convergence, possible overfitting, and the effectiveness of early stopping. Training RMSE is approximated using the square root of the average training-batch MSE.


In [ ]:

history_files = sorted(HISTORY_DIR.glob("*.csv"))

if history_files:
    history = pd.concat(
        [
            pd.read_csv(path).assign(run_file=path.name)
            for path in history_files
        ],
        ignore_index=True,
    )

    print(f"Loaded {len(history_files)} training histories.")
    print("History columns:", history.columns.tolist())
    display(history.head())
else:
    history = pd.DataFrame()
    print("No training histories found.")


In [ ]:

if not seeds.empty and not history.empty:
    selected = seeds.sort_values("val_RMSE").iloc[0]

    file_name = Path(
        str(selected["history_file"]).replace("\\", "/")
    ).name

    run = history[
        history["run_file"] == file_name
    ].sort_values("epoch")

    if not run.empty:
        fig, ax = plt.subplots(figsize=(8, 4))

        ax.plot(
            run["epoch"],
            np.sqrt(run["train_loss"]),
            label="Approx. Train RMSE"
        )

        ax.plot(
            run["epoch"],
            run["val_RMSE"],
            label="Validation RMSE"
        )

        ax.set_xlabel("Epoch")
        ax.set_ylabel("RMSE")
        ax.set_title("BiLSTM Learning Curve — Stage 9")
        ax.legend()
        ax.grid(alpha=0.3)

        plt.tight_layout()
        plt.show()
    else:
        print("Matching training history not found.")
else:
    print("Stage 9 results or training history unavailable.")



The selected configuration uses 64 embedding dimensions, 128 hidden dimensions, 0.2 dropout, and 2 LSTM layers, which achieved the strongest validation performance among the tested architectures. 

